In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df_csv = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Workspace/Users/wavexe2537@cwsgear.com/ecommerce-databricks-pipeline/data/orders.csv")

display(df_csv)

In [0]:
from pyspark.sql import functions as F

# 1. Count the incoming records
print("Total records:",df_csv.count())

# 2. Check the schema
df_csv.printSchema()

# 3. Check for duplicate order IDs
duplicate_orders = (
    df_csv.groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate order IDs:", duplicate_orders.count())

display(duplicate_orders)

# 4. Check for null values in each column
null_checks = df_csv.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df_csv.columns
])

display(null_checks)


In [0]:
spark.sql("""
    SELECT
        current_catalog() AS catalog_name,
        current_schema() AS schema_name
""").show(truncate=False)

In [0]:
# Define the Bronze table name
bronze_table = "bronze_orders"

# Add ingestion metadata while preserving the source columns
bronze_df = df_csv.withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
)

# Write the data as a managed Delta table
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(bronze_table)

print(f"Successfully loaded {bronze_df.count()} records into {bronze_table}")

In [0]:
saved_bronze_df = spark.read.table("bronze_orders")

print("Bronze record count:", saved_bronze_df.count())

display(saved_bronze_df)